# F6 — LoRA fine-tune (`lora_ft`, `lora_ft_constrained`)

Trains a LoRA adapter for `Qwen/Qwen3-1.7B` on the teacher-labeled `train.jsonl`, publishes it
to the Hugging Face Hub, and runs the two fine-tuned arms over the same 300 `eval_gold`
documents the baseline used.

**Before running:**

1. Settings → Accelerator → **`GPU T4 x2`** (the code uses `cuda:0` only — SPEC §2.2).
2. Settings → Internet **on**.
3. Add Data → the **`sxl-data`** dataset (`train.jsonl`, `dev.jsonl`, `eval_gold.jsonl`).
4. Add-ons → Secrets → add **`HF_TOKEN`** with write access, for publishing the adapter.
5. Set `COMMIT_SHA` and `ADAPTER_REPO` in the cells below.

**`/kaggle/tmp` disappears when the session ends.** The HF cache and the live checkpoints
live there for the space (~60 GB, vs 20 GB on `/kaggle/working`). Every save is mirrored to
`/kaggle/working/ckpt_latest/`, which *is* the notebook's persisted output — that mirror is
what `--resume-from-checkpoint auto` picks up in a fresh session. If you look only in
`/kaggle/tmp/ckpt` after a session dies you will conclude the run was lost when it was not.

**Budget: under 7 GPU-hours** (SPEC §6.6). Do not start the full run without a passing smoke
run. **Stop the session manually when it finishes — idle sessions burn quota.**

In [ ]:
# HF_HOME must be set BEFORE anything imports huggingface_hub: the cache path is
# frozen into module constants at import time. The default lives on /kaggle/working,
# which is only 20 GB and is the persisted notebook output — a 3.4 GB model plus
# checkpoint staging would fill it. /kaggle/tmp is ~60 GB of scratch (SPEC §2.2).
import os

os.environ["HF_HOME"] = "/kaggle/tmp/hf"
os.makedirs("/kaggle/tmp/hf", exist_ok=True)

# Training this model at batch 2 x 2048 OOMs on a 16 GB T4 with ~4.9 GB sitting
# "reserved but unallocated" — fragmentation, not demand. The caching allocator
# grabs fixed-size segments, and the large short-lived tensors here (the LM head
# produces batch x seq x ~152k vocab logits) leave holes nothing else fits in.
# Expandable segments let a block grow in place instead. Set in the kernel so the
# `!sxl gpu train` subprocesses inherit it; it must be set before torch starts.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import time

SESSION_STARTED = time.time()  # GPU-hour accounting; printed after each stage

In [ ]:
# Pin a commit SHA, never a branch (SPEC §2.4): a mid-session push must not change
# what a running notebook is executing.
COMMIT_SHA = "0000000000000000000000000000000000000000"  # <- paste `git rev-parse HEAD`

# Shape check, not a placeholder comparison: a find-and-replace over this cell
# would rewrite both copies of a placeholder string and silently disarm the guard.
import re

assert re.fullmatch(r"[0-9a-f]{40}", COMMIT_SHA)
assert set(COMMIT_SHA) != {"0"}

REPO = "https://github.com/RazaAli1010/schema-extract-lab"

# `schema-extract-lab`, not `sxl` — that is the DISTRIBUTION name from pyproject.toml.
#
# No unsloth (rejected, SPEC §5.3 — ~1.2-1.5x at 1.7B and a messy transformers 5.x
# negotiation), no flash-attn (needs sm80), no vLLM (banned, SPEC §5.3).
!pip install -q "schema-extract-lab[gpu] @ git+{REPO}@{COMMIT_SHA}"

# Three packages Kaggle preinstalls that our pinned stack cannot coexist with.
# None of them is used by this project; all three have to be absent rather than
# merely unused, because each is reached through an eager availability check that
# turns a version mismatch into a hard failure somewhere unrelated.
#
# - torchvision / torchaudio are built against Kaggle's torch (2.10.0). Our pinned
#   torch==2.13.0 leaves their compiled ops unregistered, so importing torchvision
#   raises `operator torchvision::nms does not exist` — and transformers imports it
#   eagerly via `is_torchvision_available()`, which surfaces as a baffling
#   `Could not import module 'Qwen3ForCausalLM'`.
# - torchao 0.10.0 is below peft 0.20's minimum of 0.16.0, and peft's
#   `is_torchao_available()` RAISES on an old version instead of returning False.
#   Its LoRA dispatcher calls that helper for every module it wraps, so the run
#   dies inside `get_peft_model` with an ImportError about a library we never use.
#
# RESTART THE SESSION after this cell the first time: a failed torchvision import
# leaves broken entries in sys.modules that uninstalling does not clear. Installed
# packages survive the restart, so the cell above becomes a fast no-op.
!pip uninstall -q -y torchvision torchaudio torchao

In [ ]:
# Version banner (SPEC §5.7). Printed into the saved output so a stale Kaggle base
# image is visible in the artifact rather than being a mystery six weeks later.
import importlib.metadata as md

import torch

for pkg in (
    "torch", "transformers", "trl", "peft", "accelerate",
    "bitsandbytes", "datasets", "outlines", "schema-extract-lab",
):
    try:
        print(f"{pkg:>20} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:>20} NOT INSTALLED")

name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print(f"\n{name}  capability={capability}  n_gpus={torch.cuda.device_count()}")

# Fail loudly rather than quietly training on a P100: sm_75 is what forces fp16 +
# sdpa, and every number downstream is labelled "Tesla T4".
assert capability == (7, 5), f"expected a T4 (7, 5), got {capability} on {name}"

# The trl/transformers API probe. SPEC §6.3 was verified on 2026-07-30 and this
# area of the trl API has moved repeatedly, so settle it here rather than
# discovering it three hours in. `assistant_only_loss` is the documented remedy if
# eval_loss plateaus high (F6 §Implementation notes) — check it exists before use.
import dataclasses
import inspect

import trl

print(f"\nSFTTrainer.__init__{inspect.signature(trl.SFTTrainer.__init__)}\n")
fields = {f.name for f in dataclasses.fields(trl.SFTConfig)}
for field in (
    "max_length", "max_seq_length", "model_init_kwargs",
    "assistant_only_loss", "quantization_config", "loss_type",
):
    print(f"  SFTConfig.{field}: {field in fields}")
assert "max_length" in fields, "trl API moved; SPEC §6.3 needs re-verifying"

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# The package is a pip install here, so `config.ROOT` points into site-packages and
# every default path is wrong. Pass all of them explicitly (see config.py).
#
# The mount point is discovered rather than hard-coded: depending on how a dataset
# is attached, Kaggle mounts it at /kaggle/input/<slug>/ OR at
# /kaggle/input/datasets/<owner>/<slug>/, and guessing wrong costs a session.
import glob

matches = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
assert matches, "sxl-data is not attached — use Add Input in the sidebar"
assert len(matches) == 1, f"train.jsonl found in several places: {matches}"

DATA = os.path.dirname(matches[0])
TRAIN = f"{DATA}/train.jsonl"
DEV = f"{DATA}/dev.jsonl"
GOLD = f"{DATA}/eval_gold.jsonl"

ADAPTER = "/kaggle/working/adapter"        # persisted
STATS = "/kaggle/working/train_stats.json"  # persisted, committed to results/
CKPT = "/kaggle/tmp/ckpt"                   # EPHEMERAL — dies with the session
MIRROR = "/kaggle/working/ckpt_latest"      # persisted; what `auto` resumes from
OUT = "/kaggle/working/predictions"

ADAPTER_REPO = "<hf_user>/qwen3-1.7b-jobpost-lora"  # <- set your HF username
assert "<" not in ADAPTER_REPO, "set ADAPTER_REPO to your own Hub repo id"

os.makedirs(OUT, exist_ok=True)
for path in (TRAIN, DEV, GOLD):
    assert os.path.exists(path), path

# HF_TOKEN from Kaggle Secrets. `publish()` reads it from the environment.
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

print(f"DATA = {DATA}")
!wc -l {TRAIN} {DEV} {GOLD}

# Split isolation, checked here as well as inside `train()` — before a single
# minute of GPU time is spent. This also validates every `gold` target against the
# schema: training on malformed JSON teaches the model to emit malformed JSON.
from sxl.gpu.train_lora import assert_no_leakage
from sxl.io import read_jsonl

assert_no_leakage(list(read_jsonl(TRAIN)), list(read_jsonl(DEV)), list(read_jsonl(GOLD)))
print("no leakage; every target validates")

In [ ]:
# The cell that earns its keep, against the REAL tokenizer.
#
# (a) Train/inference parity. Qwen3 renders an assistant message inside the message
#     list differently from an assistant turn opened by add_generation_prompt=True,
#     enable_thinking=False — the latter emits an empty <think></think>. That is why
#     `render_example` concatenates the inference prefix instead of handing the
#     assistant turn to apply_chat_template. Print both and prove the prefix matches.
# (b) Sequence length. A truncated JSON target teaches the model to emit truncated
#     JSON, which shows up much later as a mysteriously low schema_valid_rate.
from transformers import AutoTokenizer

from sxl.config import TRAIN_MAX_LENGTH
from sxl.gpu.runner import render_prompt
from sxl.gpu.train_lora import render_example, token_length_stats
from sxl.prompts import build_ft_prompt

tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
rows = list(read_jsonl(TRAIN))

prefix = render_prompt(tok, build_ft_prompt(rows[0]["text"]))
example = render_example(rows[0], tok)
print("prefix tail:", repr(prefix[-140:]))
print("example tail:", repr(example[-180:]))
assert example.startswith(prefix), "train/inference chat prefix drifted (SPEC §3.7)"
print("\ntrain and inference render the identical prefix")

stats = token_length_stats(rows, tok)
print(stats)
assert stats["p95"] <= TRAIN_MAX_LENGTH, (
    f"p95 {stats['p95']} > {TRAIN_MAX_LENGTH}: re-run the training cells with "
    "--max-length 3072 --batch-size 1 rather than letting targets truncate."
)

In [ ]:
# SMOKE: 32 rows, 5 optimizer steps, ~5 minutes. A finite loss, ~1% trainable, no
# OOM. **Do not run the next cell until this passes** — a full run is 1.5-3 hours
# and the failures this catches (fp16 NaN, fp16 gradient unscaling, peft not
# attaching) all show up in the first few steps.
#
# One long line, not a backslash continuation: `!` cells are handed to the shell by
# IPython and line-joining behaviour there is not worth betting a session on.
!sxl gpu train --limit 32 --max-steps 5 --train {TRAIN} --dev {DEV} --gold {GOLD} --out /kaggle/working/_smoke_adapter --stats-out /kaggle/working/_smoke_stats.json --checkpoint-dir /kaggle/tmp/ckpt_smoke --mirror-dir ""

# IPython does NOT raise when a `!` command exits non-zero, it just prints the
# traceback and carries on. Without this check the next line dies on a missing
# file, which sends you looking for a path bug instead of reading the real error
# a few lines up.
assert _exit_code == 0, f"`sxl gpu train` exited {_exit_code} — the error is printed above"

import json
import math

smoke = json.load(open("/kaggle/working/_smoke_stats.json"))
print(json.dumps(smoke, indent=2))

assert math.isfinite(smoke["final_train_loss"]), "fp16 diverged — see the playbook in the next cell"
assert 0.1 < smoke["trainable_pct"] < 5.0, smoke["trainable_pct"]
print(f"\nsmoke OK — peak VRAM {smoke['peak_vram_gb']} GB of 16")
print(f"elapsed: {(time.time() - SESSION_STARTED) / 3600:.2f} GPU-hours")

In [ ]:
# FULL RUN. 4500 rows x 2 epochs at effective batch 32 = ~281 steps. At the
# ~31 s/step the smoke measured, budget 2.5-3.5 hours.
#
# TWO epochs, not the TRAIN_EPOCHS=3 of the F6 context deltas. Three projected to
# ~7.2 h of training alone against an under-7h budget for the whole feature,
# inference included. `load_best_model_at_end` on eval_loss bounds the cost of
# stopping earlier, and F8 reports the epoch count rather than implying three.
#
# --batch-size 1 --grad-accum 16 instead of the config's 2 x 8. The EFFECTIVE
# batch is unchanged (1 x 16 x 2 GPUs = 32, exactly what the smoke ran), so no
# training math moves -- this only halves how many sequences are resident at once.
# The smoke peaked at 10.0 GB of 14.56 and still logged an allocator mapping
# failure with 13 MB free, and that was on sequences of at most 1311 tokens; the
# full corpus goes to max_length=2048, and activations scale with length. Weights
# are ~3.4 GB of that peak, so the remaining ~6.6 GB grows by roughly half and
# lands past the card. Cheaper to halve the micro-batch than to OOM at step 200.
#
# Resumable across sessions: re-running this cell after a session death picks up
# from /kaggle/working/ckpt_latest via `--resume-from-checkpoint auto`.
#
# Watch the logged loss for NaN (fp16 on a T4 is numerically touchier than bf16).
# If it goes NaN in the first 50 steps, in this order and no other (F6 §notes):
#   1. confirm bf16=False and max_grad_norm=0.3 in the echoed config
#   2. re-run with --lr 1e-4
#   3. only then --load-in-4bit
# NEVER "fix" it by switching to bf16: a T4 cannot do bf16 (SPEC §2.3).
#
# Kaggle's `GPU T4 x2` makes both cards visible and Trainer wraps the model in
# DataParallel on its own, doubling the examples per step. Left alone -- it is
# free speed and F7's latency numbers are single-T4 either way -- but
# train_stats.json records `n_gpus` and the REAL `effective_batch_size`, so the
# artifact says what ran rather than what was requested.
!sxl gpu train --epochs 2 --batch-size 1 --grad-accum 16 --train {TRAIN} --dev {DEV} --gold {GOLD} --out {ADAPTER} --stats-out {STATS} --checkpoint-dir {CKPT} --mirror-dir {MIRROR} --resume-from-checkpoint auto --push-to {ADAPTER_REPO}

assert _exit_code == 0, f"`sxl gpu train` exited {_exit_code} — the error is printed above"

print(f"elapsed: {(time.time() - SESSION_STARTED) / 3600:.2f} GPU-hours (budget: < 7)")

In [ ]:
# Training acceptance criteria, then both inference arms.
d = json.load(open(STATS))
print(json.dumps(d, indent=2))
assert 0.1 < d["trainable_pct"] < 5.0, d["trainable_pct"]
assert d["dtype"] == "float16" and "T4" in d["gpu_name"]
assert math.isfinite(d["best_eval_loss"])

# Loaded from the HUB, not from {ADAPTER}: that is what discharges "the adapter
# loads via PeftModel.from_pretrained in a fresh session".
#
# No --train flag. The fine-tuned arms have no exemplars, so `sxl gpu predict`
# does not ask for that file — its absence here is the proof the branch works.
!sxl gpu predict --arm lora_ft --adapter {ADAPTER_REPO} --gold {GOLD} --out {OUT}/lora_ft.jsonl

# Outlines-constrained, same short prompt. Sequential and slower per token, so
# budget appreciably longer than the arm above.
!sxl gpu predict --arm lora_ft_constrained --adapter {ADAPTER_REPO} --gold {GOLD} --out {OUT}/lora_ft_constrained.jsonl

In [ ]:
# Inference acceptance criteria, asserted in the artifact itself.
ARMS = ("lora_ft", "lora_ft_constrained")
gold_ids = [json.loads(line)["doc_id"] for line in open(GOLD, encoding="utf-8")]

for arm in ARMS:
    with open(f"{OUT}/{arm}.jsonl", encoding="utf-8") as fh:
        records = [json.loads(line) for line in fh]

    # Hard contract — these are what F4 depends on, so they stay assertions.
    assert len(records) == 300, (arm, len(records))
    assert [r["doc_id"] for r in records] == gold_ids, f"{arm}: doc_ids drifted from eval_gold"
    assert not any("<think>" in r["raw_output"] for r in records), f"{arm}: thinking mode leaked"

    n_valid = sum(r["schema_valid"] for r in records)
    n_trunc = sum(r["completion_tokens"] >= 512 for r in records)
    median_completion = sorted(r["completion_tokens"] for r in records)[150]
    print(
        f"{arm:>22}  valid {n_valid}/300 ({n_valid / 3:.1f}%)  "
        f"truncated {n_trunc}  median completion tokens {median_completion}"
    )

# `lora_ft_constrained` was EXPECTED to reach >= 0.99. F5 measured 0.91 for the
# constrained baseline, because the grammar permits an arbitrarily long
# `required_skills` list and a model that loops runs past MAX_NEW_TOKENS, leaving a
# truncated object that will not parse. A fine-tune emits shorter lists so this may
# well clear the bar — but it is reported rather than asserted, because halting here
# would discard a valid measurement (SPEC §1.1). F8 states the measured rate.
constrained = [json.loads(line) for line in open(f"{OUT}/lora_ft_constrained.jsonl")]
rate = sum(r["schema_valid"] for r in constrained) / len(constrained)
print(f"\nlora_ft_constrained schema_valid_rate = {rate:.3f}  (expected >= 0.99)")
if rate < 0.99:
    trunc = [r for r in constrained if not r["schema_valid"] and r["completion_tokens"] >= 512]
    print(f"  MISSED. {len(trunc)} of the invalid rows are truncation at max_new_tokens.")

print(f"\nelapsed: {(time.time() - SESSION_STARTED) / 3600:.2f} GPU-hours (budget: < 7)")
!ls -la {OUT} {ADAPTER}

## Back on the laptop

Download from this notebook's output:

- `predictions/lora_ft.jsonl` and `predictions/lora_ft_constrained.jsonl` → `artifacts/predictions/`
- `train_stats.json` → `results/`

Then:

```bash
sxl metrics score --arm lora_ft
sxl metrics score --arm lora_ft_constrained
sxl metrics compare
```

**Never commit the adapter**, `*.safetensors`, `*.bin`, or any checkpoint directory (SPEC §2.4).
The Hub is the artifact store; `train_stats.json` and the two prediction files are the only
things that come back.

`lora_ft` is expected above `base_fewshot`'s macro-F1 of 0.5704. **If it is not, that is the
result** (SPEC §1.1) — report it, investigate loss masking (`assistant_only_loss`, only if the
version-banner cell confirmed the flag exists) and epoch count, and do not quietly weaken the
baseline.

**Now stop the session** (Run → Stop session) — idle sessions keep burning the weekly quota.